About Dataset
Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

asin - ID of the product, like B000FA64PK

helpful - helpfulness rating of the review - example: 2/3.

overall - rating of the product.

reviewText - text of the review (heading).

reviewTime - time of the review (raw).

reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN

reviewerName - name of the reviewer.

summary - summary of the review (description).

unixReviewTime - unix timestamp.

Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

**Inspiration**

-Sentiment analysis on reviews.

-Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.

-Fake reviews/ outliers.

-Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).

**Best Practises**

Preprocessing And Cleaning

Train Test Split

BOW, TFIDF, Word2vec

Train ML algorithms

In [1]:
#Load dataset

import pandas as pd
data = pd.read_csv("/content/all_kindle_review.csv")
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [2]:
#only review text and rating is required

data = data[['reviewText','rating']]
data

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4
...,...,...
11995,Valentine cupid is a vampire- Jena and Ian ano...,4
11996,I have read all seven books in this series. Ap...,5
11997,This book really just wasn't my cuppa. The si...,3
11998,"tried to use it to charge my kindle, it didn't...",1


In [4]:
#missing values
data.isnull().sum()

,0
reviewText,0
rating,0


In [4]:
data['rating'].unique()

array([3, 5, 4, 2, 1])

In [5]:
data['rating'].value_counts()

,count
rating,
5,3000
4,3000
3,2000
2,2000
1,2000


In [3]:
#data preprocessing

#Positive review - 1 nd negative review - 0
data['rating'] = data['rating'].apply(lambda x:0 if x<3 else 1)

In [ ]:
data['rating'].unique()

array([1, 0])

In [8]:
data['rating'].value_counts()

,count
rating,
1,8000
0,4000


In [4]:
#Lowercase all reviews

data['reviewText'] = data['reviewText'].str.lower()

In [5]:
import nltk
import re
nltk.download('stopwords')

from nltk.corpus import stopwords


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [6]:
from bs4 import BeautifulSoup

In [7]:
#Data cleaning

#Removing special characters
data['reviewText'] = data['reviewText'].apply(lambda x:re.sub('[^a-zA-Z0-9]+',' ',x))

#Remove stopwords
data['reviewText'] = data['reviewText'].apply(lambda x:' '.join([word for word in x.split() if word not in stopwords.words('english')]))

#Remove url
data['reviewText'] = data['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))

#Remove html tags
data['reviewText'] = data['reviewText'].apply(lambda x: BeautifulSoup(x, 'html.parser').get_text())   #html parser converts x into beautiful soup object


In [8]:
data

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four books expecting 34 con...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1
...,...,...
11995,valentine cupid vampire jena ian another vampi...,1
11996,read seven books series apocalyptic adventure ...,1
11997,book really cuppa situation man capturing woma...,1
11998,tried use charge kindle even register charging...,0


In [9]:
#Lemmatization

from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [10]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [11]:
data['reviewText'] = data['reviewText'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))
#split words of each sentence and then lemmatize them and then join them back

In [12]:
data.head()

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four book expecting 34 conc...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [13]:
#train test split

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(data['reviewText'],data['rating'],test_size = 0.20)

In [14]:
X_train

,reviewText
5337,movie prompted buy book found quite engrossing...
10312,thing loved book moment hero attracted heroine...
1309,normally write review think enough book share ...
728,left dead trust ever much different book guy o...
9979,guess cookbook would okay beginning vegetarian...
...,...
10177,vivi andrew always good witty comical escape m...
2399,amazing story glad read highly recommend anyon...
2936,interesting mix mystery magic unassuming light...
969,good easy read first hate h start soften towar...


In [14]:
#Create BOW model

from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()

In [26]:
#Training on BOW

X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [27]:
#Create TF-IDF model

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [28]:
#Training on TF-IDF

X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

Gaussian Naive Baye's

In [30]:
#Model training

from sklearn.naive_bayes import GaussianNB
import numpy as np

model_bow = GaussianNB().fit(X_train_bow,y_train)
model_tfidf = GaussianNB().fit(X_train_tfidf,y_train)


In [33]:
y_pred_bow = model_bow.predict(X_test_bow)
y_pred_tfidf = model_tfidf.predict(X_test_tfidf)

In [34]:
#Performance metrics

from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))
print(confusion_matrix(y_test,y_pred_bow))
print(classification_report(y_test,y_pred_bow))

BOW accuracy:  0.5833333333333334
[[513 272]
 [728 887]]
              precision    recall  f1-score   support

           0       0.41      0.65      0.51       785
           1       0.77      0.55      0.64      1615

    accuracy                           0.58      2400
   macro avg       0.59      0.60      0.57      2400
weighted avg       0.65      0.58      0.60      2400



In [35]:
print("TF-IDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))
print(confusion_matrix(y_test,y_pred_tfidf))
print(classification_report(y_test,y_pred_tfidf))

TF-IDF accuracy:  0.58625
[[501 284]
 [709 906]]
              precision    recall  f1-score   support

           0       0.41      0.64      0.50       785
           1       0.76      0.56      0.65      1615

    accuracy                           0.59      2400
   macro avg       0.59      0.60      0.57      2400
weighted avg       0.65      0.59      0.60      2400



Word2Vec Model

In [1]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 657.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 8.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatib

In [15]:
import gensim
from gensim.models import Word2Vec,KeyedVectors

In [16]:
import gensim.downloader as api

wv = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [21]:
"""model = gensim.models.Word2Vec(reviews)"""
wv.index_to_key   #display vocabulary

['</s>',
 'in',
 'for',
 'that',
 'is',
 'on',
 '##',
 'The',
 'with',
 'said',
 'was',
 'the',
 'at',
 'not',
 'as',
 'it',
 'be',
 'from',
 'by',
 'are',
 'I',
 'have',
 'he',
 'will',
 'has',
 '####',
 'his',
 'an',
 'this',
 'or',
 'their',
 'who',
 'they',
 'but',
 '$',
 'had',
 'year',
 'were',
 'we',
 'more',
 '###',
 'up',
 'been',
 'you',
 'its',
 'one',
 'about',
 'would',
 'which',
 'out',
 'can',
 'It',
 'all',
 'also',
 'two',
 'after',
 'first',
 'He',
 'do',
 'time',
 'than',
 'when',
 'We',
 'over',
 'last',
 'new',
 'other',
 'her',
 'people',
 'into',
 'In',
 'our',
 'there',
 'A',
 'she',
 'could',
 'just',
 'years',
 'some',
 'U.S.',
 'three',
 'million',
 'them',
 'what',
 'But',
 'so',
 'no',
 'like',
 'if',
 'only',
 'percent',
 'get',
 'did',
 'him',
 'game',
 'back',
 'because',
 'now',
 '#.#',
 'before',
 'company',
 'any',
 'team',
 'against',
 'off',
 'This',
 'most',
 'made',
 'through',
 'make',
 'second',
 'state',
 'well',
 'day',
 'season',
 'says',
 'w

In [33]:
len(wv)

3000000

In [23]:
wv.similar_by_word('tree')

[('trees', 0.8293122053146362),
 ('pine_tree', 0.7622087001800537),
 ('oak_tree', 0.731893002986908),
 ('evergreen_tree', 0.6926872730255127),
 ('fir_tree', 0.6917218565940857),
 ('willow_tree', 0.6845874190330505),
 ('pine_trees', 0.6824266910552979),
 ('maple_tree', 0.6803498268127441),
 ('sycamore_tree', 0.6681810617446899),
 ('tress', 0.6547872424125671)]

In [24]:
wv[0].shape

(300,)

In [18]:
# Tokenize the sentences by splitting them into words
X_train_tokens = X_train.apply(lambda x: x.split())
X_test_tokens = X_test.apply(lambda x: x.split())

In [25]:
X_train_tokens

,reviewText
5337,"[movie, prompted, buy, book, found, quite, eng..."
10312,"[thing, loved, book, moment, hero, attracted, ..."
1309,"[normally, write, review, think, enough, book,..."
728,"[left, dead, trust, ever, much, different, boo..."
9979,"[guess, cookbook, would, okay, beginning, vege..."
...,...
10177,"[vivi, andrew, always, good, witty, comical, e..."
2399,"[amazing, story, glad, read, highly, recommend..."
2936,"[interesting, mix, mystery, magic, unassuming,..."
969,"[good, easy, read, first, hate, h, start, soft..."


In [36]:
import numpy as np

def vectorize_documents(data_tokens, model):
    """Converts a list of tokenized documents to a list of averaged vectors."""
    vectorized_data = []
    vector_size = model.vector_size  # This will be 300 for the Google model

    for doc_tokens in data_tokens:
        # Get vectors for words in the document that are in the model's vocabulary
        word_vectors = [model[word] for word in doc_tokens if word in model]

        if len(word_vectors) > 0:
            # Average the word vectors to get a single document vector
            avg_vector = np.mean(word_vectors, axis=0)
            vectorized_data.append(avg_vector)
        else:
            # If a document has no words in the vocabulary, append a zero vector
            vectorized_data.append(np.zeros(vector_size))

    return np.array(vectorized_data)    #returns a list of avg vectors for each sentence of the entire document passed in this func

In [27]:
X_train_wv = vectorize_documents(X_train_tokens, wv)
X_test_wv = vectorize_documents(X_test_tokens, wv)

In [28]:
print("\nShape of vectorized training data:", X_train_wv.shape)
print("Shape of vectorized testing data:", X_test_wv.shape)


Shape of vectorized training data: (9600, 300)
Shape of vectorized testing data: (2400, 300)


In [29]:
#Model training

from sklearn.naive_bayes import GaussianNB
import numpy as np

model_wv = GaussianNB().fit(X_train_wv,y_train)

In [30]:
y_pred_wv = model_wv.predict(X_test_wv)
print(y_pred_wv)

[1 0 1 ... 0 1 0]


In [32]:
#Performance metrics

from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

print("Word2Vec accuracy: ",accuracy_score(y_test,y_pred_wv))
print(confusion_matrix(y_test,y_pred_wv))
print(classification_report(y_test,y_pred_wv))

Word2Vec accuracy:  0.7433333333333333
[[ 652  176]
 [ 440 1132]]
              precision    recall  f1-score   support

           0       0.60      0.79      0.68       828
           1       0.87      0.72      0.79      1572

    accuracy                           0.74      2400
   macro avg       0.73      0.75      0.73      2400
weighted avg       0.77      0.74      0.75      2400



Accuracy improved from bow and tf-idf